# 00 · 项目总览：要做什么，为什么重做，怎么做

## 这一章在整个项目里的位置

```
[00 总览] → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 第一个基线 → 09 起：工具、agent …
```

这一章没有代码，只把整件事讲清楚：要做一个什么东西，为什么要从头重做，按什么顺序做。后面每一章开头都有上面这一行，告诉你现在走到哪了。

## 背景：为什么要重做

**OtoVCE 论文已经投出去了，但它有两个问题：**

1. **它不是真正的 agent。** 它是一条固定流程：规则打分 → 查文献 → 大模型读文献打分 → 汇总。每一步做什么、按什么顺序，都写死在代码里，大模型只在其中一格里干活
2. **它只做耳聋。** 一个病种

**代码也拿不出手。** 找 AI agent 应用工程师的工作，面试官会打开你的代码库看。旧项目是这样的：

| 现象 | 数字 |
|---|---|
| 提交很少 | 5 个月只有 9 次提交；最后两个月的工作根本没存进记录 |
| 几代代码叠在一起 | 约 390 个 Python 文件、近 8 万行，step1–5、src、otovce、otovce_v2、curator 五套并存 |
| 代码互相找不到 | 88 个文件开头要"手动指路"（`sys.path`）才能找到别的代码 |
| 数据混进代码记录 | 22 个基因组索引文件进了 git |
| 自动检查只管一角 | 只检查 otovce_v2 一个目录 |

底子是好的：防泄漏、预注册、对照组这些评测上的讲究都是真的。缺的是**工程上的收纳**，不是能力。

**所以这次**：换一个更大的题目，从第一天起按公司的标准做。

## 要做成什么：一个罕见病诊断 agent

输入一个病人的症状，agent **自己决定下一步查什么**，最后给出最可能的几个病和致病基因，每一条都附上证据和出处。

```
病人的症状（一段文字）
        │
        ▼
 ┌─────────── agent：看目前知道什么，决定下一步 ───────────┐
 │  症状 → 标准医学术语（HPO）                              │
 │  症状 → 可能的病有哪些                                   │
 │  查某个病的典型表现、遗传方式、致病基因                   │
 │  核对：病人和这个病，哪些对得上、哪些缺、哪些矛盾         │
 │  查文献（自动挡住病例自己的来源论文）                     │
 └──────────────────────────────────────────────────────────┘
        │
        ▼
最可能的 5 个病 + 致病基因 + 每条的证据和出处 + 建议下一步做什么检查
```

### agent 和固定流程，差在哪

这是整个项目的核心，也是旧论文的短板：

| | 固定流程（旧论文） | agent（这次） |
|---|---|---|
| 下一步做什么 | 写死在代码里 | 模型看目前的情况自己决定 |
| 遇到没想到的情况 | 照样走完流程 | 可以换个方向再查 |
| 大模型的角色 | 流水线上的一格 | 指挥整个过程 |

**agent 不一定比固定流程好**，这要靠评测来证明。所以会把它和 4 种做法放在同一批病例上比：

| 对照 | 做法 | 回答什么问题 |
|---|---|---|
| 传统方法 | 只按症状打分，不用大模型 | 不用大模型能做到多好 |
| 大模型直接猜 | 把症状交给大模型，一次回答 | 不给工具能做到多好 |
| 固定流程 | 工具按固定顺序跑一遍，最后大模型汇总 | **为什么需要 agent** |
| DeepRare | 已发表的同类系统 | 和别人比怎么样 |

## 怎么做：路线

| 步 | 做什么 | 做完能看到 | 章节 | 状态 |
|---|---|---|---|---|
| 0 | 存档旧项目 | 论文那一版随时能找回 | 01 | ✅ |
| 1 | 搭新项目的骨架 | 标准项目、自动检查、GitHub | 02–04 | ✅ |
| 2 | **先做"尺子"：评测** | 一万例考题、判分、第一个准确率数字 | 05 起 | 进行中 |
| 3 | 做工具 | agent 能调用的各种查询，每个都有测试 | | |
| 4 | 做 agent | 看它一步步查了什么、为什么；和固定流程比 | | |
| 5 | 做成服务 | 换台电脑一条命令跑起来。**到这里就能拿去面试** | | |
| 6 | 大规模评测 | 一万例全量结果、和 DeepRare 对比：论文主表 | | |
| 7 | 交付 | 论文、项目介绍、演示视频 | | |

### 为什么先做尺子，再做 agent

没有评测，改了一版 agent，你不知道它是变好了还是变坏了，只能凭感觉挑几个例子看。公司里做 agent 也是这个顺序：**先定好怎么打分（eval），再开发**。招聘要求里写的 "evaluation"、"LLMOps"，指的就是这一块。

## 公司里怎么管代码

公司里一个项目很多人同时改，靠人记住"这个改动会不会弄坏别处"是不可能的。所以代码库都有一套**自动把关**的流程：

```
你改代码 → 提交前自动检查 → 推到 GitHub → 自动跑全部检查 → 有人看过 → 合并进正本
            (pre-commit)                    (CI)              (Pull Request)
```

归结起来就三件事：

| | 做法 | 得到什么 | 哪一章 |
|---|---|---|---|
| **存** | 每次改动存一个版本，正本放在 GitHub | 不怕丢，能退回，看得出谁改了什么 | 01、02、04 |
| **检** | 机器自动检查每一个改动 | 低级错误进不来 | 03、04 |
| **审** | 改动先放一边，检查绿了、人看过，才并入正本 | 正本永远是好的 | 04 |

面试时问"你们的开发流程是什么样的"，答的就是这条线。

## 常见的两个问题

### agent 有没有 Databricks 那样的平台？

有，比如 MLflow（Databricks 自带的就是它）、LangSmith、Langfuse。但和 Databricks 一样：**代码还是自己写，平台管写完以后的事**：把 agent 跑起来、记下它的每一步、给结果打分、对比不同版本。你在 qafi-data-platform 里写的 dbt 代码也是代码，Databricks 只是运行和管理它的地方。

我们用 **MLflow**：从 07 号起，每次评测都记在它里面；后面 agent 每一步的追踪也用它。它开源、能装在本机、断网也能用，换成公司的服务器只改一个地址。

真正不写代码的是 Dify、Coze 这类拖拽平台：快，但做法改不了。工程师岗位主要考的是写代码这一种。

### 这里用的东西，都是公司里的标准做法吗？

几乎都是。分三类：
- **所有公司都一样的**：git、GitHub、合并请求、自动检查（CI）、测试、依赖锁定。01–04 号全是这一类
- **有几家可选、我们选最常见的**：agent 框架（LangGraph，同类还有 CrewAI、AutoGen）、工具接口（MCP）、实验追踪（MLflow，同类还有 Weights & Biases、LangSmith）。换一家公司可能换牌子，但做的事一样
- **只为了学习才有的**：这本笔记本身

## 一句话

把一个只做耳聋、按固定流程跑的研究项目，重做成一个**能自己决定查什么的罕见病诊断 agent**；先装好公司级的工程流程和评测，再做 agent。

## 面试怎么说

> "I rebuilt a research codebase into an engineered agent project: version control and CI from day one, an evaluation harness with leakage control before any agent code, then a tool-using agent compared against fixed-pipeline and single-call baselines."

**下一章 01 · 存档旧项目**：动手之前，先把论文那一版钉住，把两个月没存的工作收好。